In [1]:
!pip install datasets transformers torch pandas -q
!pip install --upgrade datasets torchvision accelerate -q
!pip uninstall torchaudio -y

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2

In [2]:
!pip install --upgrade torch torchvision transformers datasets -q

#Imports e Definição do Modelo

In [3]:
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer

# Definição do modelo
MODEL_NAME = "distilbert-base-uncased"


print(f"Carregando tokenizador para: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Carregando tokenizador para: distilbert-base-uncased


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

#Carregamento e Amostragem

In [4]:
# Carrega o arquivo do Kaggle
dataset = load_dataset('csv', data_files="/content/IMDB Dataset.csv", split='train')

# Corte estratégico de dados para salvar tempo de processamento
# Selecionando 10.000 reviews aleatórias das 50.000 originais
dataset = dataset.shuffle(seed=42).select(range(15000))

print(f"Tamanho do dataset após corte: {len(dataset)} exemplos.")

Generating train split: 0 examples [00:00, ? examples/s]

Tamanho do dataset após corte: 15000 exemplos.


#Mapeamento e Tokenização

In [5]:
# 1. Converte 'positive' para 1 e 'negative' para 0
def map_labels(example):
    example['labels'] = 1 if example['sentiment'] == 'positive' else 0
    return example

dataset = dataset.map(map_labels)

# 2. Função de tokenização limitando a 128 tokens
def tokenize_function(examples):
    return tokenizer(
        examples["review"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

print("Iniciando tokenização em lote...")
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("Tokenização concluída!")

Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Iniciando tokenização em lote...


Map:   0%|          | 0/15000 [00:00<?, ? examples/s]

Tokenização concluída!


#Divisão e Preparação para o PyTorch

In [6]:
# Separa 80% para Treino e 20% para Teste/Validação
train_test_split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

# Divide os 20% restantes igualmente: Validação, Teste
test_val_split = train_test_split['test'].train_test_split(test_size=0.5, seed=42)

# Agrupa tudo em um único objeto DatasetDict
final_datasets = DatasetDict({
    'train': train_test_split['train'],
    'validation': test_val_split['train'],
    'test': test_val_split['test']
})

# Define o formato esperado pelos tensores do modelo
final_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Dataset IMDB pronto para o Trainer!")
print(final_datasets)

Dataset IMDB pronto para o Trainer!
DatasetDict({
    train: Dataset({
        features: ['review', 'sentiment', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 12000
    })
    validation: Dataset({
        features: ['review', 'sentiment', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1500
    })
    test: Dataset({
        features: ['review', 'sentiment', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1500
    })
})


#Função de Métricas (Accuracy, Precision, Recall, F1)

In [7]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    # Calcula precision, recall e f1 usando a média binária (já que é positivo/negativo)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)

    return {
        'accuracy': acc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    }

#Experimento 1

In [ ]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. CONGELA AS CAMADAS (Exp 1)
# ==========================================
for param in model.base_model.parameters():
    param.requires_grad = False
print("Experimento 1: Apenas classificador liberado.")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================

EPOCAS_ATUAIS = 5

training_args = TrainingArguments(
    output_dir="./resultados_exp1",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 1 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 1): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 1 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Experimento 1: Apenas classificador liberado.

Iniciando treinamento do Experimento 1 por 5 épocas...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.647182,0.553078,0.773333,0.720000,0.857143,0.782609
2,0.523287,0.483729,0.788667,0.753512,0.826331,0.788243
3,0.497278,0.456556,0.792667,0.772666,0.799720,0.785960
4,0.477916,0.447988,0.800667,0.776298,0.816527,0.795904
5,0.478752,0.445252,0.803333,0.779706,0.817927,0.798360


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tempo total de treinamento (Exp 1): 4.57 minutos

=== MÉTRICAS FINAIS EXP 1 (CONJUNTO DE TESTE) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.478752,0.457299,5,0.785333,0.781746,0.790107,0.785904


loss: 0.4573
accuracy: 0.7853
precision: 0.7817
recall: 0.7901
f1: 0.7859


#Experimento 2

In [15]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. CONGELA AS CAMADAS (Exp 2 - Parcial)
# ==========================================
# Primeiro, congela tudo
for param in model.base_model.parameters():
    param.requires_grad = False

# Depois, libera apenas as duas últimas camadas
for name, param in model.named_parameters():
    if "layer.4" in name or "layer.5" in name:
        param.requires_grad = True
print("Experimento 2: Fine-tuning parcial liberado (Últimas camadas).")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================
EPOCAS_ATUAIS = 5

training_args = TrainingArguments(
    output_dir="./resultados_exp2",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 2 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 2): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 2 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Experimento 2: Fine-tuning parcial liberado (Últimas camadas).

Iniciando treinamento do Experimento 2 por 5 épocas...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.430374,0.337677,0.854000,0.810540,0.904762,0.855063
2,0.320691,0.327068,0.866000,0.859748,0.858543,0.859145
3,0.287227,0.336291,0.860667,0.805320,0.932773,0.864374
4,0.255569,0.339913,0.870667,0.828283,0.918768,0.871182
5,0.239208,0.329659,0.873333,0.842932,0.901961,0.871448


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tempo total de treinamento (Exp 2): 7.07 minutos

=== MÉTRICAS FINAIS EXP 2 (CONJUNTO DE TESTE) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.239208,0.356637,5,0.848667,0.865358,0.824866,0.844627


loss: 0.3566
accuracy: 0.8487
precision: 0.8654
recall: 0.8249
f1: 0.8446


#Experimento 3

In [16]:
import time
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments

# ==========================================
# 1. RECARREGA O MODELO DO ZERO
# ==========================================
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ==========================================
# 2. LIBERA TODAS AS CAMADAS (Exp 3)
# ==========================================
for param in model.parameters():
    param.requires_grad = True
print("Experimento 3: Fine-tuning completo (Todas as camadas liberadas).")

# ==========================================
# 3. RECRIA O TRAINER E TREINA
# ==========================================
EPOCAS_ATUAIS = 5

training_args = TrainingArguments(
    output_dir="./resultados_exp3",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCAS_ATUAIS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=final_datasets["train"],
    eval_dataset=final_datasets["validation"],
    compute_metrics=compute_metrics,
)

print(f"\nIniciando treinamento do Experimento 3 por {EPOCAS_ATUAIS} épocas...")
start_time = time.time()
trainer.train()
end_time = time.time()
print(f"Tempo total de treinamento (Exp 3): {(end_time - start_time) / 60:.2f} minutos")

# ==========================================
# 4. AVALIAÇÃO FINAL NO TESTE
# ==========================================
print("\n=== MÉTRICAS FINAIS EXP 3 (CONJUNTO DE TESTE) ===")
resultados_teste = trainer.evaluate(eval_dataset=final_datasets["test"])
for chave, valor in resultados_teste.items():
    if chave.startswith("eval_"):
        print(f"{chave.replace('eval_', '')}: {valor:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Experimento 3: Fine-tuning completo (Todas as camadas liberadas).

Iniciando treinamento do Experimento 3 por 5 épocas...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.408587,0.297860,0.876667,0.837803,0.918768,0.876420
2,0.241411,0.323547,0.880000,0.852243,0.904762,0.877717
3,0.149808,0.439851,0.880000,0.850394,0.907563,0.878049
4,0.088047,0.546775,0.877333,0.841495,0.914566,0.876510
5,0.055001,0.576294,0.880000,0.855053,0.900560,0.877217


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Tempo total de treinamento (Exp 3): 12.47 minutos

=== MÉTRICAS FINAIS EXP 3 (CONJUNTO DE TESTE) ===


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.055001,0.323285,5,0.853333,0.835025,0.879679,0.856771


loss: 0.3233
accuracy: 0.8533
precision: 0.8350
recall: 0.8797
f1: 0.8568
